In [17]:
from langchain_community.document_loaders import JSONLoader
from langchain_core.documents import Document
from itertools import groupby
import json

# ── 1. Load JSON manual ──────────────────────────────────────────
with open("Qwen_json_20260313_krtrbraab.json", "r", encoding="utf-8") as f:
    data = json.load(f)  # list of dict

print(f"Jumlah dokumen: {len(data)}")
print(data[0])

# ── 2. Buat Document per ayat dengan metadata yang benar ─────────
docs = []
for item in data:
    docs.append(Document(
        page_content=item["content"],
        metadata={
            "id":    item["id"],
            "pasal": item["pasal"],
            "ayat":  item["ayat"],
            "bab":   item["bab"]
        }
    ))
print(docs)

Jumlah dokumen: 71
{'id': 'psl_1_ay_1', 'pasal': 1, 'ayat': 1, 'bab': 'BAB I - BENTUK DAN KEDAULATAN', 'content': 'Negara Indonesia ialah Negara kesatuan yang berbentuk Republik.'}
[Document(metadata={'id': 'psl_1_ay_1', 'pasal': 1, 'ayat': 1, 'bab': 'BAB I - BENTUK DAN KEDAULATAN'}, page_content='Negara Indonesia ialah Negara kesatuan yang berbentuk Republik.'), Document(metadata={'id': 'psl_1_ay_2', 'pasal': 1, 'ayat': 2, 'bab': 'BAB I - BENTUK DAN KEDAULATAN'}, page_content='Kedaulatan adalah di tangan rakyat, dan dilakukan sepenuhnya oleh Majelis Permusyawaratan Rakyat.'), Document(metadata={'id': 'psl_2_ay_1', 'pasal': 2, 'ayat': 1, 'bab': 'BAB II - MAJELIS PERMUSYAWARATAN RAKYAT'}, page_content='Majelis Permusyawaratan Rakyat terdiri atas anggota-anggota Dewan Perwakilan Rakyat, ditambah dengan utusan-utusan dari daerah-daerah dan golongan-golongan, menurut aturan yang ditetapkan dengan undang-undang.'), Document(metadata={'id': 'psl_2_ay_2', 'pasal': 2, 'ayat': 2, 'bab': 'BAB II

In [18]:
# ── 3. Group per pasal ───────────────────────────────────────────
docs_sorted = sorted(docs, key=lambda x: x.metadata["pasal"])

grouped_docs = []
for pasal_num, ayats in groupby(docs_sorted, key=lambda x: x.metadata["pasal"]):
    ayats = list(ayats)
    combined_text = " ".join([a.page_content for a in ayats])
    grouped_docs.append(Document(
        page_content=combined_text,
        metadata={
            "pasal":      pasal_num,
            "bab":        ayats[0].metadata["bab"],
            "ayat_range": f"{ayats[0].metadata['ayat']}-{ayats[-1].metadata['ayat']}"
        }
    ))

print(f"Jumlah grup pasal: {len(grouped_docs)}")
print(grouped_docs[0].page_content)
print(grouped_docs[0].metadata)

Jumlah grup pasal: 37
Negara Indonesia ialah Negara kesatuan yang berbentuk Republik. Kedaulatan adalah di tangan rakyat, dan dilakukan sepenuhnya oleh Majelis Permusyawaratan Rakyat. Panitia Persiapan Kemerdekaan Indonesia mengatur dan menyelenggarakan kepindahan pemerintahan kepada Pemerintah Indonesia. Dalam enam bulan sesudah akhirnya peperangan Asia Timur Raya, Presiden Indonesia mengatur dan menyelenggarakan segala hal yang ditetapkan dalam Undang-Undang Dasar ini. Dalam enam bulan sesudah Majelis Permusyawaratan Rakyat dibentuk, Majelis itu bersidang untuk menetapkan Undang-Undang Dasar.
{'pasal': 1, 'bab': 'BAB I - BENTUK DAN KEDAULATAN', 'ayat_range': '1-2'}


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Split dari grouped_docs (per pasal), bukan per ayat
texts = text_splitter.split_documents(grouped_docs)
print(f"Jumlah chunks: {len(texts)}")
print(texts[0].page_content)
print(texts[0].metadata)

Jumlah chunks: 42
Negara Indonesia ialah Negara kesatuan yang berbentuk Republik. Kedaulatan adalah di tangan rakyat, dan dilakukan sepenuhnya oleh Majelis Permusyawaratan Rakyat. Panitia Persiapan Kemerdekaan Indonesia mengatur dan menyelenggarakan kepindahan pemerintahan kepada Pemerintah Indonesia. Dalam enam bulan sesudah akhirnya peperangan Asia Timur Raya, Presiden Indonesia mengatur dan menyelenggarakan segala hal yang ditetapkan dalam Undang-Undang Dasar ini
{'pasal': 1, 'bab': 'BAB I - BENTUK DAN KEDAULATAN', 'ayat_range': '1-2'}


In [22]:
texts[0]

Document(metadata={'pasal': 1, 'bab': 'BAB I - BENTUK DAN KEDAULATAN', 'ayat_range': '1-2'}, page_content='Negara Indonesia ialah Negara kesatuan yang berbentuk Republik. Kedaulatan adalah di tangan rakyat, dan dilakukan sepenuhnya oleh Majelis Permusyawaratan Rakyat. Panitia Persiapan Kemerdekaan Indonesia mengatur dan menyelenggarakan kepindahan pemerintahan kepada Pemerintah Indonesia. Dalam enam bulan sesudah akhirnya peperangan Asia Timur Raya, Presiden Indonesia mengatur dan menyelenggarakan segala hal yang ditetapkan dalam Undang-Undang Dasar ini')

In [20]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os

embeddings = HuggingFaceEmbeddings(
    model_name="/Users/muhammadzuamaalamin/Documents/fintunellm/model/multilingual-e5-small",
    model_kwargs={"device": "cpu"}
)

db_path = "faiss_indexe5"

if not os.path.exists(db_path):
    vectorstore = FAISS.from_documents(texts, embeddings)
    vectorstore.save_local(db_path)
else:
    vectorstore = FAISS.load_local(
        db_path,
        embeddings,
        allow_dangerous_deserialization=True
    )


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [26]:
query = "bunyi pasal 1 uud 1945  ?"
results = vectorstore.similarity_search_with_score(query, k=5)

for i, (doc, score) in enumerate(results, 1):
    print(f"\nResult {i} | score={score:.4f}")
    print(doc.page_content[:400])


Result 1 | score=0.3485
Bahasa negara ialah Bahasa Indonesia.

Result 2 | score=0.3634
. Dalam enam bulan sesudah Majelis Permusyawaratan Rakyat dibentuk, Majelis itu bersidang untuk menetapkan Undang-Undang Dasar.

Result 3 | score=0.3689
. Segala badan negara dan peraturan yang ada masih langsung berlaku, selama belum diadakan yang baru menurut Undang-Undang Dasar ini.

Result 4 | score=0.3722
Bendera Negara Indonesia ialah Sang Merah Putih.

Result 5 | score=0.3773
Pemerintah memajukan kebudayaan nasional Indonesia.


In [27]:
from langchain.retrievers import BM25Retriever, EnsembleRetriever

bm25 = BM25Retriever.from_documents(texts)
bm25.k = 5

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25, vector_retriever],
    weights=[0.5, 0.5]
)

In [31]:
query = "Apa bentuk negara Indonesia menurut UUD 1945?  ?"

docs = hybrid_retriever.get_relevant_documents(query)

for i, (doc, score) in enumerate(results, 1):
    print(f"\nResult {i} | score={score:.4f}")
    print(doc.page_content[:400])



Result 1 | score=0.3485
Bahasa negara ialah Bahasa Indonesia.

Result 2 | score=0.3634
. Dalam enam bulan sesudah Majelis Permusyawaratan Rakyat dibentuk, Majelis itu bersidang untuk menetapkan Undang-Undang Dasar.

Result 3 | score=0.3689
. Segala badan negara dan peraturan yang ada masih langsung berlaku, selama belum diadakan yang baru menurut Undang-Undang Dasar ini.

Result 4 | score=0.3722
Bendera Negara Indonesia ialah Sang Merah Putih.

Result 5 | score=0.3773
Pemerintah memajukan kebudayaan nasional Indonesia.
